# Carregando dados e imports

In [9]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

ROOT = Path("..").resolve()

df = pd.read_csv(ROOT / "data" / "processed" / "covid.csv")
X = df.drop(columns="ÓBITO")
y = df["ÓBITO"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)
print(f"Treino {len(X_tr):,} | Validação {len(X_val):,} | Teste {len(X_test):,} | taxa de óbito: {y.mean():.2%}")

Treino 671,088 | Validação 167,772 | Teste 209,715 | taxa de óbito: 7.34%


# Treinando o modelo

In [10]:
rng = np.random.default_rng(42)
idx_pos = y_tr.index[y_tr == 1]
idx_neg = rng.choice(y_tr.index[y_tr == 0], size=len(idx_pos), replace=False)
idx_bal = np.concatenate([idx_pos, idx_neg])
print(f"Treino balanceado: {len(idx_bal):,} registros (1:1)")

modelo = CatBoostClassifier(
    iterations=200,
    learning_rate=0.03,
    depth=5,
    loss_function="Logloss",
    random_seed=42,
    allow_writing_files=False,
    train_dir=tempfile.gettempdir(),
    verbose=100,
)
modelo.fit(X_tr.loc[idx_bal], y_tr.loc[idx_bal]);

Treino balanceado: 98,486 registros (1:1)
0:	learn: 0.6641701	total: 13.9ms	remaining: 2.77s
100:	learn: 0.3005845	total: 807ms	remaining: 791ms
199:	learn: 0.2919464	total: 1.58s	remaining: 0us


# Ajustando o limiar de decisão

In [11]:
proba_val = modelo.predict_proba(X_val)[:, 1]
prec, rec, limiares = precision_recall_curve(y_val, proba_val)
f1 = 2 * prec * rec / (prec + rec + 1e-12)
limiar = float(limiares[int(np.nanargmax(f1[:-1]))])
print(f"Limiar escolhido: {limiar:.3f} (F1 validação: {np.nanmax(f1[:-1]):.3f})")

Limiar escolhido: 0.846 (F1 validação: 0.626)


# Avaliando no conjunto de teste

In [12]:
proba_teste = modelo.predict_proba(X_test)[:, 1]
pred_teste = (proba_teste >= limiar).astype(int)

print(f"AUC-ROC (teste): {roc_auc_score(y_test, proba_teste):.4f}")
print(f"PR-AUC  (teste): {average_precision_score(y_test, proba_teste):.4f}\n")
print(classification_report(y_test, pred_teste, target_names=["Sobreviveu", "óbito"]))
print("Matriz de confusão:")
pd.DataFrame(
    confusion_matrix(y_test, pred_teste),
    index=["Real: Sobreviveu", "Real: óbito"],
    columns=["Pred: Sobreviveu", "Pred: óbito"],
)

AUC-ROC (teste): 0.9467
PR-AUC  (teste): 0.6697

              precision    recall  f1-score   support

  Sobreviveu       0.97      0.96      0.97    194327
       óbito       0.59      0.65      0.62     15388

    accuracy                           0.94    209715
   macro avg       0.78      0.81      0.79    209715
weighted avg       0.94      0.94      0.94    209715

Matriz de confusão:


,Pred: Sobreviveu,Pred: óbito
Real: Sobreviveu,187492,6835
Real: óbito,5379,10009
